# Ablation Study — CLPsych 2026 Task 1

Generates 4 test-set submissions for the paper's ablation tables, sharing one inference run per model.

**Ablation 1 — Prompting Strategy** (ensemble fixed, S3-merge style: Qwen elements only + averaged presence)
- `ablation1_p1_only` — unified call only
- `ablation1_p1_p2` — unified + adaptive-focused
- (P1+P2+P3 already submitted as S3)

**Ablation 3 — Ensemble Merge** (full P1+P2+P3 pipeline)
- `ablation3_qwen_alone` — Qwen single-model
- `ablation3_mistral_alone` — Mistral single-model
- (S2 union and S3 Qwen-only already submitted)

**Total: 4 new Codabench submissions.**

## Run order
1. Run cells 1–6 (Qwen).
2. Restart runtime.
3. Re-run cells 1–4, then run cell 7 (Mistral).
4. Run cell 8 to build all 4 submission zips.

In [ ]:
# 1. Setup
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader  # optional: confirms a GPU is available
%pip install -q -r requirements.txt

In [ ]:
MODEL_1 = 'Qwen/Qwen3.5-27B'
MODEL_2 = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
TEMPERATURE = 0.1
MAX_TOKENS = 100
RAG_K = 8
RAG_POOL = 20
RAG_ALPHA = 0.6

In [ ]:
  #patch

  import os
  os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
  import rag_index, gc, torch
  from sentence_transformers import SentenceTransformer

  rag_index._model_cache.clear()
  gc.collect(); torch.cuda.empty_cache()

  def _get_model_safe(model_name="Alibaba-NLP/gte-multilingual-base"):
      if model_name not in rag_index._model_cache:
          m = SentenceTransformer(model_name, trust_remote_code=True, device='cpu')
          m.max_seq_length = 512
          # Fix: position_ids buffer не инициализирован в этой версии transformers.
          emb = m[0].auto_model.embeddings
          if hasattr(emb, 'position_ids') and emb.position_ids is not None:
              n = emb.position_ids.shape[-1]
              emb.position_ids = torch.arange(n, dtype=torch.long).unsqueeze(0)
              print(f'Fixed position_ids: shape={emb.position_ids.shape}, range=[0..{n-1}]')
          else:
              # Buffer не существует — создаём
              emb.register_buffer('position_ids', torch.arange(8192, dtype=torch.long).unsqueeze(0), persistent=False)
              print('Created position_ids buffer.')
          rag_index._model_cache[model_name] = m
      return rag_index._model_cache[model_name]

  def _embed_texts_safe(texts, model_name="Alibaba-NLP/gte-multilingual-base"):
      model = _get_model_safe(model_name)
      return model.encode(texts, normalize_embeddings=True, show_progress_bar=True,
                          batch_size=8, convert_to_numpy=True)

  rag_index._get_model = _get_model_safe
  rag_index._embed_texts = _embed_texts_safe
  print('Patched.')

In [ ]:
from data_loader import load_all_timelines
from rag_index import RAGIndex
from prompt_builder_rag import build_messages, build_messages_adaptive, build_messages_per_element
from output_parser import parse_llm_output
from tqdm import tqdm
import json, gc, torch
from pathlib import Path
from vllm import LLM, SamplingParams

# Build RAG index from train
train_posts = load_all_timelines('train_tasks12')
rag_index_real = RAGIndex([p for p in train_posts if p.has_evidence])

# Load test
test_posts = load_all_timelines('test_tasks12nolabels')
print(f'Test posts: {len(test_posts)}')

# Build P1 prompts (unified) and P3 prompts (per-element A). P2 is built later from P1 outputs.
test_messages = []
test_rag_cache = []
for p in tqdm(test_posts, desc='P1 prompts'):
    rag = rag_index_real.retrieve(p.text, k=RAG_K, exclude_timeline=p.timeline_id,
                                  candidate_pool=RAG_POOL, diversity_alpha=RAG_ALPHA)
    test_messages.append(build_messages(p.text, rag))
    test_rag_cache.append(rag)

test_messages_A = []
for p in tqdm(test_posts, desc='P3 prompts'):
    rag_A = rag_index_real.retrieve_for_element(p.text, 'A', k=RAG_K, exclude_timeline=p.timeline_id)
    test_messages_A.append(build_messages_per_element(p.text, 'A', rag_A))

In [ ]:
# Free GPU before loading vLLM
import rag_index
rag_index._model_cache.clear()
gc.collect()
torch.cuda.empty_cache()
!nvidia-smi --query-gpu=memory.used --format=csv
print('Ready for vLLM.')

In [ ]:
def run_pipeline(model_name):
    """Run full 3-call pipeline. Returns three lists of parsed outputs (P1, P2, P3)."""
    print(f'\nLoading {model_name}...')
    llm = LLM(model_name, dtype='float16', max_model_len=16384, tensor_parallel_size=1)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    chat_kwargs = {'enable_thinking': False} if 'qwen' in model_name.lower() else {}

    # P1 — Unified
    out_p1 = llm.chat(test_messages, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p1 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p1]

    # P2 — Adaptive-focused (depends on P1 maladaptive context)
    msgs_p2 = [build_messages_adaptive(p.text, test_rag_cache[i], parsed_p1[i]['maladaptive'])
               for i, p in enumerate(test_posts)]
    out_p2 = llm.chat(msgs_p2, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p2 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p2]

    # P3 — Per-element Affect (independent)
    out_p3 = llm.chat(test_messages_A, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p3 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p3]

    del llm; gc.collect(); torch.cuda.empty_cache()
    return parsed_p1, parsed_p2, parsed_p3

qwen_p1, qwen_p2, qwen_p3 = run_pipeline(MODEL_1)

Path('outputs').mkdir(exist_ok=True)
with open('outputs/ablation_qwen_raw.json', 'w') as f:
    json.dump({'p1': qwen_p1, 'p2': qwen_p2, 'p3': qwen_p3}, f)
print('Saved outputs/ablation_qwen_raw.json')
print('\n>>> Restart runtime, re-run cells 1–4, then run the next cell. <<<')

---
## After runtime restart
Re-run cells 1, 2, 3, 4 (skip cell 5 — vLLM not loaded yet), then run the next cell.

In [ ]:
  # Проверить что занимает место
  !du -sh ~/.cache/huggingface/hub/* 2>/dev/null | sort -h
  !df -h /

In [ ]:
  !rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3.5-27B
  !df -h /

In [ ]:
  #patch

  import os
  os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
  print('vLLM будет использовать spawn (избегает CUDA fork bug).')

In [ ]:
# Run Mistral (after restart). Loads Qwen raw outputs from disk.
import rag_index
rag_index._model_cache.clear()
gc.collect(); torch.cuda.empty_cache()

with open('outputs/ablation_qwen_raw.json') as f:
    qwen_raw = json.load(f)
qwen_p1, qwen_p2, qwen_p3 = qwen_raw['p1'], qwen_raw['p2'], qwen_raw['p3']
print(f'Loaded Qwen raw: {len(qwen_p1)} P1, {len(qwen_p2)} P2, {len(qwen_p3)} P3')

# (run_pipeline is defined in cell 6 — re-define here in case kernel restarted)
def run_pipeline(model_name):
    print(f'\nLoading {model_name}...')
    llm = LLM(model_name, dtype='float16', max_model_len=16384, tensor_parallel_size=1)
    params = SamplingParams(max_tokens=MAX_TOKENS, temperature=TEMPERATURE)
    chat_kwargs = {'enable_thinking': False} if 'qwen' in model_name.lower() else {}
    out_p1 = llm.chat(test_messages, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p1 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p1]
    msgs_p2 = [build_messages_adaptive(p.text, test_rag_cache[i], parsed_p1[i]['maladaptive'])
               for i, p in enumerate(test_posts)]
    out_p2 = llm.chat(msgs_p2, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p2 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p2]
    out_p3 = llm.chat(test_messages_A, sampling_params=params, chat_template_kwargs=chat_kwargs)
    parsed_p3 = [parse_llm_output(o.outputs[0].text.strip()) for o in out_p3]
    del llm; gc.collect(); torch.cuda.empty_cache()
    return parsed_p1, parsed_p2, parsed_p3

mistral_p1, mistral_p2, mistral_p3 = run_pipeline(MODEL_2)
with open('outputs/ablation_mistral_raw.json', 'w') as f:
    json.dump({'p1': mistral_p1, 'p2': mistral_p2, 'p3': mistral_p3}, f)
print('Saved outputs/ablation_mistral_raw.json')

In [ ]:
# Build all 4 ablation submissions from cached raw outputs.
import json, zipfile, sys, subprocess
from pathlib import Path
from submission_formatter import save_submission

with open('outputs/ablation_qwen_raw.json') as f:
    qwen_raw = json.load(f)
with open('outputs/ablation_mistral_raw.json') as f:
    mistral_raw = json.load(f)
qwen_p1, qwen_p2, qwen_p3 = qwen_raw['p1'], qwen_raw['p2'], qwen_raw['p3']
mistral_p1, mistral_p2, mistral_p3 = mistral_raw['p1'], mistral_raw['p2'], mistral_raw['p3']

def build_within_model(unified, adaptive=None, per_a=None):
    """Combine raw call outputs into a single-model prediction.
    Mirrors the submission pipeline: maladaptive + adaptive-presence from unified;
    adaptive elements from P2 if available; Affect overridden by P3 if available."""
    merged = {
        'maladaptive': {'elements': dict(unified['maladaptive']['elements']),
                        'presence': unified['maladaptive']['presence']},
        'adaptive': {'elements': dict(adaptive['adaptive']['elements']) if adaptive else dict(unified['adaptive']['elements']),
                     'presence': unified['adaptive']['presence']},
    }
    if per_a is not None:
        for v in ('adaptive', 'maladaptive'):
            merged[v]['elements'].pop('A', None)
            if 'A' in per_a[v]['elements']:
                merged[v]['elements']['A'] = per_a[v]['elements']['A']
    for v in ('adaptive', 'maladaptive'):
        if not merged[v]['elements']:
            merged[v]['presence'] = 1
    return merged

def merge_ensemble_s3(qpred, mpred):
    """S3-merge: Qwen elements only, averaged presence (with the {1,2}=>2 rule)."""
    merged = {'maladaptive': {'elements': {}, 'presence': 1},
              'adaptive': {'elements': {}, 'presence': 1}}
    for v in ('maladaptive', 'adaptive'):
        pq, pg = qpred[v]['presence'], mpred[v]['presence']
        if pq == pg: merged[v]['presence'] = pq
        elif {pq, pg} == {1, 2}: merged[v]['presence'] = 2
        else: merged[v]['presence'] = round((pq + pg) / 2)
        merged[v]['elements'] = dict(qpred[v]['elements'])
        if not merged[v]['elements']: merged[v]['presence'] = 1
    return merged

configs = {
    'ablation1_p1_only': ('P1 only · ensemble (S3-merge)',
        lambda i: merge_ensemble_s3(
            build_within_model(qwen_p1[i]),
            build_within_model(mistral_p1[i]))),
    'ablation1_p1_p2': ('P1+P2 · ensemble (S3-merge)',
        lambda i: merge_ensemble_s3(
            build_within_model(qwen_p1[i], qwen_p2[i]),
            build_within_model(mistral_p1[i], mistral_p2[i]))),
    'ablation3_qwen_alone': ('Qwen alone · P1+P2+P3',
        lambda i: build_within_model(qwen_p1[i], qwen_p2[i], qwen_p3[i])),
    'ablation3_mistral_alone': ('Mistral alone · P1+P2+P3',
        lambda i: build_within_model(mistral_p1[i], mistral_p2[i], mistral_p3[i])),
}

Path('outputs').mkdir(exist_ok=True)
if 'official_eval' not in sys.path:
    sys.path.insert(0, 'official_eval')
for name, (description, fn) in configs.items():
    print(f'\n=== {description} ===')
    final = []
    for i, p in enumerate(test_posts):
        final.append({'timeline_id': p.timeline_id, 'post_id': p.post_id, 'prediction': fn(i)})

    raw_path = f'outputs/{name}_raw.json'
    pred_path = f'outputs/{name}_task1_pred.json'
    with open(raw_path, 'w') as f:
        json.dump(final, f, indent=2)
    save_submission(final, pred_path)

    result = subprocess.run(
        [sys.executable, 'official_eval/validate_submission.py',
         '--task1', pred_path, '--test-dir', 'test_tasks12nolabels'],
        capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode != 0:
        print('STDERR:', result.stderr)

    zip_path = f'outputs/{name}.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        zf.write(pred_path, 'task1_pred.json')

print('\nDone! 4 submission zips saved to outputs/.')